# Review API detections

Pick sites on a map, take a random sample of the detections there, then listen to each clip and say whether the detection is right.

**You need:** a `.env` file in the repo folder with your `API_TOKEN` (to find detections) and AWS keys (to play the audio). See the README.

**Run it** in JupyterLab, one step at a time: pick sites on the map in step 1 before you run step 2. The review tool in step 3 doesn't work in VS Code.

In [1]:
import os
import re
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from dotenv import load_dotenv
from jupyter_bioacoustic import BioacousticAnnotator

# Work from the repo folder, so every path here and in annotator_config/ starts there.
if Path.cwd().name == "notebooks":
    os.chdir("..")

load_dotenv(override=True)  # API_TOKEN, and the AWS keys the review tool uses to play audio from S3
API_URL = "https://api.wildlifesoundhub.org"
COOKIES = {"__Secure-authjs.session-token": os.environ["API_TOKEN"]}


def query_api(path, timeout=120, **params):
    """GET `{API_URL}{path}?{params}` with the session cookie and return the JSON body."""
    r = requests.get(f"{API_URL}{path}", params=params, cookies=COOKIES, timeout=timeout)
    r.raise_for_status()
    return r.json()


query_api("/users/me")  # a 401 error here means API_TOKEN in .env has expired
print("Connected")

Connected


## 1. Pick sites

Click a site to select it (it turns red). Click it again to unselect it. Hover over a site to see its name and number of recordings.

In [2]:
deployments = pd.DataFrame(query_api("/deployments/")).rename(columns={"id": "deployment_id", "placename": "site"})
deployments[["lon", "lat"]] = [g["coordinates"] for g in deployments.geometry]
recordings = pd.DataFrame(query_api("/recordings/")).merge(deployments[["deployment_id", "site", "lat", "lon"]], on="deployment_id")
sites = recordings.groupby("site", as_index=False).agg(lat=("lat", "mean"), lon=("lon", "mean"), recordings=("id", "size"))

class SiteMap(go.FigureWidget):
    # Panning or zooming sends a "map._derived" update that plotly reports as an error. Drop it.
    def plotly_relayout(self, relayout_data, **kwargs):
        super().plotly_relayout({k: v for k, v in relayout_data.items() if not k.endswith("._derived")}, **kwargs)

fig = SiteMap(px.scatter_map(
    sites, lat="lat", lon="lon", hover_name="site", hover_data=["recordings"],
    center=dict(lat=sites.lat.mean(), lon=sites.lon.mean()), zoom=5, height=600,
))
fig.update_traces(marker_size=10, selected_marker_color="crimson")
fig.update_layout(clickmode="event", margin=dict(l=0, r=0, t=0, b=0)) #modebar_remove=["select", "lasso"], 
picked = set()  # rows of `sites` selected on the map

def handle_selection(trace, points, state):
    # points.point_inds contains all points currently inside the box/lasso area
    # Use symmetric difference if you want to toggle them, 
    # or just use `set(points.point_inds)` to match the selection exactly
    global picked
    picked = set(points.point_inds)
    
    # Update the trace to highlight selected points
    trace.selectedpoints = sorted(picked) if picked else None

# Connect the selection event (handles both Box Select and Lasso Select)
fig.data[0].on_selection(handle_selection)

def toggle_site(trace, points, state):
    picked.symmetric_difference_update(points.point_inds)
    trace.selectedpoints = sorted(picked) or None  # None = nothing selected; [] would grey out every site

fig.data[0].on_click(toggle_site)
fig

SiteMap({
    'data': [{'customdata': {'bdata': ('LABAAEAAQAAuAC4AEAAOAC4ALQAeAC' ... 'AwAC4AEAAQABAAEAAQABAAEAAeAB4A'),
                             'dtype': 'i2',
                             'shape': '582, 1'},
              'hovertemplate': ('<b>%{hovertext}</b><br><br>lat' ... '{customdata[0]}<extra></extra>'),
              'hovertext': array(['ACWA1', 'ACWA2', 'ACWA3', ..., 'ZASL4', 'ZAY1', 'ZAY3'],
                                 shape=(582,), dtype=object),
              'lat': {'bdata': ('kTAryQ6TRECUWbVSHZlEQPnj7lp3lk' ... '1c2jaNQkDV8oIHhYZCQHZEcu6EjUJA'),
                      'dtype': 'f8'},
              'legendgroup': '',
              'lon': {'bdata': ('eQ0ZRHhEXsCeDI6Sl0NewAfQ7/tHRF' ... 'udWqKCXsAtiYxCIYRewFK5lLSSgV7A'),
                      'dtype': 'f8'},
              'marker': {'color': '#636efa', 'size': 10},
              'mode': 'markers',
              'name': '',
              'selected': {'marker': {'color': 'crimson'}},
              'showlegend': False,

## 2. Choose clips

Set the filters below and run the next two cells. They find every detection at your sites that passes the filters, then pick a random few per site. The same sites and filters always pick the same clips, so you can come back to them later.

Rerun both cells after changing the filters or the sites on the map.

In [3]:
SPECIES_WORDS = ["power_tools", "engine", "fireworks", "gun", "human_vocal", "human_non_vocal"]  # species with a word ending in one of these, e.g. "Pacific Treefrog" but not "Tawny Frogmouth"
MIN_CONFIDENCE = 0.4  # 0 to 1: how sure the model must be
CLIPS_PER_SITE = 1
ONE_CLIP_PER_RECORDING = True  # False allows several clips from the same recording

In [4]:
# get selected sites in list
selected_sites = sites.site.iloc[sorted(picked)].tolist()
assert selected_sites, "Select at least one site on the map above"
print("Sites:", ", ".join(selected_sites))
species_pattern = rf"({'|'.join(SPECIES_WORDS)})\b"

# query api for detections at each deployment, filtering by species and confidence
def find_detections(deployment_id):
    """Detections at one deployment (a recorder at a site) that pass the filters."""
    # The API only matches exact species names, so list the species here and fetch the ones that match.
    species = query_api("/detections/", deployment=deployment_id, confidence=MIN_CONFIDENCE, count=True)
    names = [s["common_name"] for s in species if re.search(species_pattern, s["common_name"], re.IGNORECASE)]
    rows = [d for name in names for d in query_api("/detections/", deployment=deployment_id, confidence=MIN_CONFIDENCE, common_name=name)]
    return pd.DataFrame(rows).assign(deployment_id=deployment_id)

# get detections at selected sites
deployment_site = recordings[recordings.site.isin(selected_sites)].groupby("deployment_id").site.first()
with ThreadPoolExecutor(max_workers=8) as pool:  # each API call takes 5-20 s, so run them in parallel
    hits = pd.concat(pool.map(find_detections, deployment_site.index), ignore_index=True)
assert len(hits), "No detections at these sites pass the filters"
hits["site"] = hits.deployment_id.map(deployment_site)

# filter detections at selected sites
clips = hits.sample(frac=1, random_state=0)  # shuffle, so the picks below are random
if ONE_CLIP_PER_RECORDING:
    clips = clips.drop_duplicates("recording_id")
clips = clips.groupby("site").head(CLIPS_PER_SITE).sort_values(["site", "confidence"], ascending=[True, False])
clips["recording_id"] = clips.recording_id.astype(int)  # pd.concat leaves it as float
clips["id"] = clips.recording_id.astype(str) + "_" + clips.start_time.astype(str) + "_" + clips.common_name
clips["path"] = "s3://" + clips.recording_id.map(recordings.set_index("id").path)  # the review tool plays audio from here

# save filtered detections to load in annotator tool
Path("detection_data").mkdir(exist_ok=True)
clips[["id", "site", "recording_id", "common_name", "scientific_name", "confidence", "start_time", "end_time", "path"]
      ].to_csv("detection_data/filtered_detections.csv", index=False)

print(f"Found {len(hits):,} detections in {hits.recording_id.nunique():,} recordings. Picked {len(clips)} clips.")
for site, n in clips.site.value_counts().items():
    if n < CLIPS_PER_SITE:
        print(f"Only {n} clips available at {site}")
clips.groupby(["site", "common_name"]).size().rename("clips").to_frame()

Sites: BAEC1, BAEC2, BAEC3, BAEC4, ELDO1, ELDO2, ELDO3, ELDO4, GLWA1, GLWA2, GLWA3, GLWA4, GRHO1, GRHO2, GRHO3, GRHO4, HUSY1, HUSY2, HUSY2c, HUSY3, HUSY4, IVNC1, IVNC2, IVNC3, IVNC4, MABE1, MABE2, MABE2b, MABE3, MABE4, NFN1, NFN2, NFN3, NFN3b, NFNV1, NFNV2, NFNV3, NFNV4, NFNV4b, RSWO1, RSWO2, RSWO3, RSWO4, SAHE1, SAHE2, SAHE3, SAHE4, SHC1, SHC3, SHC3b, SHRU1, SHRU2, SHRU3, SHRU4, SHSY1, SHSY2, SHSY3, SHSY4, SWA1, SWA2, SWA4, UBBWA1, UBBWA4, UNAV1, UNAV2, UNAV3, UNAV4, WOMY1, WOMY2, WOMY3, WOMY4, WOYU1, WOYU4
Found 74,004 detections in 1,123 recordings. Picked 71 clips.


,,clips
site,common_name,
BAEC1,Engine,1
BAEC2,Engine,1
BAEC3,Engine,1
BAEC4,Engine,1
ELDO1,Engine,1
...,...,...
WOMY2,Engine,1
WOMY3,Engine,1
WOMY4,Engine,1


## 3. Review

Listen to each clip and answer the questions. Answers are saved to `output/api_reviews.csv` as you go.

The questions and the columns shown are set in `annotator_config/`.

In [ ]:
ann = BioacousticAnnotator(project="annotator_config/projects/api_detections.yaml")
ann.open()